# <font color=firebrick>Econometrics — M2 MBFA</font>
## TD 2 / 6 — Applying Multivariate OLS

**Prerequisites:** Session 3 (CM) — multivariate OLS and multicollinearity.

**Format:** two exercises (Part 1), an analysis on a dataset (Part 2), and an analysis on a dataset of your choice (Part 3).

**Today's objectives:**
- Verify, on small examples, how omitting a variable biases a coefficient, and how the matrix formula for $\hat\beta$ actually works.
- Run a multivariate analysis on real data: choosing regressors, estimating, testing, and interpreting.
- Practice on a dataset you pick yourself.

---


## Part 1 — Warm-up: two small checks

### 1.1 Omitted variable bias, by hand and at scale

**Task 1.1 - Part A (by hand).** Consider the true model (through the origin, as in Sessions 1-2) $y_i=\beta_1 x_{1i}+\beta_2 x_{2i}+\varepsilon_i$ with $\beta_1=2$, $\beta_2=1$, and **no noise** ($\varepsilon_i=0$ for every $i$, so the identity below should hold *exactly*, not just approximately). Take:

| $x_{1i}$ | 1 | 2 | 3 | 4 |
|---|---|---|---|---|
| $x_{2i}$ | 2 | 1 | 4 | 3 |

1. Compute $y_i=2x_{1i}+x_{2i}$ for each $i$.
2. Compute the simple-regression (through-origin) coefficient you would get by regressing $y$ on $x_1$ **alone**: $\tilde\beta_1=\dfrac{\sum_i x_{1i}y_i}{\sum_i x_{1i}^2}$.
3. Compute $\delta_1=\dfrac{\sum_i x_{1i}x_{2i}}{\sum_i x_{1i}^2}$ (the slope from regressing $x_2$ on $x_1$), and check that $\tilde\beta_1=\beta_1+\beta_2\delta_1$ *exactly*.

<details>
<summary>Show answer</summary>

$y=(2\times1+2,\;2\times2+1,\;2\times3+4,\;2\times4+3)=(4,5,10,11)$.

$\sum_i x_{1i}y_i=1(4)+2(5)+3(10)+4(11)=4+10+30+44=88$, and $\sum_i x_{1i}^2=1+4+9+16=30$, so $\tilde\beta_1=88/30\approx2.933$.

$\sum_i x_{1i}x_{2i}=1(2)+2(1)+3(4)+4(3)=2+2+12+12=28$, so $\delta_1=28/30\approx0.933$.

$\beta_1+\beta_2\delta_1=2+1\times0.933=2.933$ - matches $\tilde\beta_1$ exactly, because with $\varepsilon_i=0$ there is no noise term left to create any gap.

</details>

**Task 1.1 - Part B (computational, at scale).** Real data always has noise, so the match won't be exact - but it should still be *close*, especially with a large sample. Complete the code below: generate a large, correlated pair $(x_1,x_2)$, build $y$ from the true model (with noise this time), and check that the simple-regression coefficient $\tilde\beta_1$ (omitting $x_2$) is close to $\beta_1+\beta_2\hat\delta_1$, where $\hat\delta_1$ is estimated from the same sample.


In [ ]:
import numpy as np

rng = np.random.default_rng(seed=7)

beta1_true, beta2_true = 2.0, 1.0
N = 20_000

x1 = rng.exponential(1, size=N)
x2 = 0.6 * x1 + rng.normal(0, 1, size=N)          # x2 is correlated with x1 by construction
eps = rng.normal(0, 1, size=N)

y = beta1_true * x1 + beta2_true * x2 + eps         # the true (2-variable) model

# TODO: compute the simple-regression coefficient beta1_tilde, omitting x2 (through-origin formula from Session 2)
beta1_tilde = ...

# TODO: compute delta1, the (through-origin) coefficient from regressing x2 on x1
delta1 = ...

print("beta1_tilde (simple regression, x2 omitted): ", beta1_tilde)
print("beta1_true + beta2_true * delta1:            ", beta1_true + beta2_true * delta1)
assert np.isclose(beta1_tilde, beta1_true + beta2_true * delta1, rtol=0.05), "These should be close - check your formulas."
print("Self-check passed!")


### 1.2 The matrix formula, by hand and with `numpy`

**Task 1.2 - Part A (by hand).** Take a tiny dataset with $N=4$ and **two regressors, no intercept** (to keep the matrix small):

| $x_{1i}$ | 1 | 2 | 1 | 2 |
|---|---|---|---|---|
| $x_{2i}$ | 1 | 1 | 2 | 2 |
| $y_i$ | 3 | 5 | 4 | 7 |

1. Compute the $2\times2$ matrix $\mathbf{X}^\top\mathbf{X}=\begin{pmatrix}\sum x_{1i}^2 & \sum x_{1i}x_{2i}\\ \sum x_{1i}x_{2i} & \sum x_{2i}^2\end{pmatrix}$ and the vector $\mathbf{X}^\top y=\begin{pmatrix}\sum x_{1i}y_i\\ \sum x_{2i}y_i\end{pmatrix}$.
2. Invert the $2\times2$ matrix by hand (recall: for $\mathbf{A}=\begin{pmatrix}a&b\\c&d\end{pmatrix}$, $\mathbf{A}^{-1}=\dfrac{1}{ad-bc}\begin{pmatrix}d&-b\\-c&a\end{pmatrix}$), and compute $\hat\beta=(\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top y$.

<details>
<summary>Show answer</summary>

$\sum x_{1i}^2=1+4+1+4=10$, $\sum x_{2i}^2=1+1+4+4=10$, $\sum x_{1i}x_{2i}=1+2+2+4=9$, so $\mathbf{X}^\top\mathbf{X}=\begin{pmatrix}10&9\\9&10\end{pmatrix}$.

$\sum x_{1i}y_i=1(3)+2(5)+1(4)+2(7)=3+10+4+14=31$, $\sum x_{2i}y_i=1(3)+1(5)+2(4)+2(7)=3+5+8+14=30$, so $\mathbf{X}^\top y=\begin{pmatrix}31\\30\end{pmatrix}$.

$\det(\mathbf{X}^\top\mathbf{X})=10\times10-9\times9=19$, so $(\mathbf{X}^\top\mathbf{X})^{-1}=\dfrac{1}{19}\begin{pmatrix}10&-9\\-9&10\end{pmatrix}$.

$\hat\beta=\dfrac{1}{19}\begin{pmatrix}10(31)-9(30)\\-9(31)+10(30)\end{pmatrix}=\dfrac{1}{19}\begin{pmatrix}40\\21\end{pmatrix}\approx\begin{pmatrix}2.105\\1.105\end{pmatrix}$.

</details>

**Task 1.2 - Part B (computational).** Complete the code below to get the same result with `numpy`, directly from the boxed formula of Section 3.


In [ ]:
import numpy as np

X = np.array([[1, 1],
              [2, 1],
              [1, 2],
              [2, 2]])
y = np.array([3, 5, 4, 7])

# TODO: compute X'X, X'y, and beta_hat = (X'X)^{-1} X'y
XtX = ...
Xty = ...
beta_hat = ...

print("X'X =\n", XtX)
print("X'y =", Xty)
print("beta_hat =", beta_hat)
assert np.allclose(beta_hat, [40/19, 21/19], atol=1e-6), "Doesn't match the hand-computed answer - check X'X, X'y, and the inverse."
print("Self-check passed!")


### 1.3 Multicollinearity, by hand and with `numpy`

**Task 1.3 - Part A (by hand).** Recall $\mathbf{X}^\top\mathbf{X}=\begin{pmatrix}10&r\\r&10\end{pmatrix}$ from Task 1.2, where $r=\sum_i x_{1i}x_{2i}$ measures how much $x_1$ and $x_2$ move together. Its inverse involves dividing by $\det(\mathbf{X}^\top\mathbf{X})=100-r^2$ (Section 3 of the CM).

1. Compute $\det(\mathbf{X}^\top\mathbf{X})=100-r^2$ for $r=0$, $r=9$, and $r=9.9$.
2. What happens to the entries of $(\mathbf{X}^\top\mathbf{X})^{-1}=\dfrac{1}{100-r^2}\begin{pmatrix}10&-r\\-r&10\end{pmatrix}$ as $r\to10$? Recall from Section 3 that $\text{Var}(\hat\beta\mid\mathbf{X})=\sigma^2(\mathbf{X}^\top\mathbf{X})^{-1}$ - what does this imply for the precision of $\hat\beta$?

<details>
<summary>Show answer</summary>

$\det=100$ for $r=0$; $\det=100-81=19$ for $r=9$; $\det=100-98.01=1.99$ for $r=9.9$ - it shrinks fast as $r$ approaches $10$ (the value at which the two columns of $\mathbf{X}$ would become proportional, i.e. perfectly collinear).

Since every entry of $(\mathbf{X}^\top\mathbf{X})^{-1}$ is divided by this shrinking determinant, they all **grow** as $r\to10$ - and so does $\text{Var}(\hat\beta\mid\mathbf{X})=\sigma^2(\mathbf{X}^\top\mathbf{X})^{-1}$. Two nearly-collinear regressors make $\hat\beta$ increasingly imprecise, even though it remains unbiased.

</details>

**Task 1.3 - Part B (computational).** Complete the code below: simulate two versions of the same model, one where $x_1$ and $x_2$ are essentially uncorrelated, one where they are highly correlated, and compare the reported standard error of $\hat\beta_1$.


In [ ]:
import statsmodels.api as sm

rng = np.random.default_rng(seed=99)
N = 500
beta1_true, beta2_true = 1.0, 1.0

def simulate_and_fit(rho):
    x1 = rng.normal(0, 1, size=N)
    # TODO: build x2 so that its correlation with x1 is approximately `rho`
    # (hint: a weighted average of x1 and a fresh, independent noise term)
    x2 = ...
    eps = rng.normal(0, 1, size=N)
    y = beta1_true * x1 + beta2_true * x2 + eps
    X = np.column_stack([x1, x2])
    return sm.OLS(y, X).fit()

res_low_corr = simulate_and_fit(rho=0.0)
res_high_corr = simulate_and_fit(rho=0.95)

print("Correlation ~0.00 :  SE(beta1_hat) =", res_low_corr.bse[0])
print("Correlation ~0.95 :  SE(beta1_hat) =", res_high_corr.bse[0])
assert res_high_corr.bse[0] > 2 * res_low_corr.bse[0], "The standard error should be much larger under high correlation - check your x2."
print("Self-check passed!")


## Part 2 — Guided analysis: CEO pay, revisited with new variables

We return to CEO compensation, but with a different (and richer) dataset: `ceosal2`, also from `wooldridge`, with $177$ observations. Unlike `ceosal1` (used in TD 1 and in Session 3), it does not have `roe` - instead it has firm size (`sales`, `mktval` = market value), CEO characteristics (`age`, `ceoten` = years as CEO, `college`, `grad`), and their logs.


In [ ]:
from wooldridge import load_data
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

ceo2 = load_data.data('ceosal2')
ceo2.head()


In [ ]:
load_data.data('ceosal2', description=True)


### 2.1 Choosing regressors

**Task 2.1.** Before regressing anything, look at the variable list above and propose a specification: which variables would you include to explain `lsalary`, and why? Write down your model (in words or as an equation) before running any code. There is no single right answer, but you should be able to justify each variable economically.

For the rest of this guided part, we will use:
$$
\log(\text{salary}_i) = \beta_0 + \beta_1\,\text{lsales}_i + \beta_2\,\text{lmktval}_i + \beta_3\,\text{ceoten}_i + \beta_4\,\text{college}_i + \varepsilon_i.
$$


### 2.2 A look before regressing

**Task 2.2.** `lsales` and `lmktval` are two different measures of firm size (sales and market value). Compute their correlation below. Given what Section 3 says about multicollinearity, what would you expect if their correlation turned out to be very close to $1$?


In [ ]:
ceo2[['lsales', 'lmktval', 'ceoten', 'age', 'college']].corr()


### 2.3 Estimating and reading the results

**Task 2.3.** Estimate the model from 2.1 with `statsmodels`, then answer:
1. Which coefficients are statistically significant at the 5% level?
2. `college` is a dummy (`=1` if the CEO attended college). Is its coefficient significant? What would you conclude - and is "not significant" the same thing as "college has no effect on CEO pay"?


In [ ]:
res2 = smf.ols('lsalary ~ lsales + lmktval + ceoten + college', data=ceo2).fit()
print(res2.summary())


### 2.4 Economic interpretation

**Task 2.4.** Pick the `ceoten` coefficient (years as CEO of the company).
1. Since `lsalary` is logged but `ceoten` is not, this coefficient is (approximately) a **semi-elasticity**: one additional year as CEO is associated with a $100\times\hat\beta_{\text{ceoten}}$% change in salary. Compute this percentage.
2. Is this economically large or small over, say, a 10-year tenure? Justify with a number.


In [ ]:
# TODO: compute the % change in salary associated with one additional year as CEO
pct_ceoten = ...
print(f"One additional year as CEO is associated with a {pct_ceoten:.2f}% change in salary.")


### 2.5 What changes if we drop a variable?

**Task 2.5.** Re-estimate the model *without* `lmktval`. Compare the coefficient on `lsales` in the two specifications. Does it change much? Relate what you see to Section 3's omitted variable bias result and to the correlation you computed in 2.2.


In [ ]:
res2_no_mktval = smf.ols('lsalary ~ lsales + ceoten + college', data=ceo2).fit()
print(res2_no_mktval.params)
print()
print(res2.params)


### 2.6 Goodness of fit, by hand and with `statsmodels`

**Task 2.6 - Part A (by hand).** Take the tiny dataset $x=(1,2,3,4)$, $y=(2,3,3,6)$, and the simple regression (with intercept) $y=\beta_0+\beta_1x+\varepsilon$.

1. Using the formulas from Section 2 of the CM, compute $\hat\beta_1$ and $\hat\beta_0$.
2. Compute the fitted values $\hat y_i$ and the residuals $y_i-\hat y_i$.
3. Compute $TSS=\sum_i(y_i-\bar y)^2$, $RSS=\sum_i(y_i-\hat y_i)^2$, and $R^2=1-RSS/TSS$.
4. Compute the adjusted $R^2$ (here $N=4$, $p=1$ regressor excluding the intercept).

<details>
<summary>Show answer</summary>

$\bar x=2.5$, $\bar y=3.5$. $\hat\beta_1=\dfrac{\sum(x_i-\bar x)(y_i-\bar y)}{\sum(x_i-\bar x)^2}=\dfrac{6}{5}=1.2$, $\hat\beta_0=3.5-1.2(2.5)=0.5$.

Fitted values: $\hat y=(1.7,\,2.9,\,4.1,\,5.3)$. Residuals: $(0.3,\,0.1,\,-1.1,\,0.7)$.

$RSS=0.3^2+0.1^2+1.1^2+0.7^2=1.8$. $TSS=(2-3.5)^2+(3-3.5)^2+(3-3.5)^2+(6-3.5)^2=9$.

$R^2=1-1.8/9=0.8$. Adjusted $R^2=1-(1-0.8)\dfrac{4-1}{4-1-1}=1-0.2\times\dfrac{3}{2}=0.7$.

</details>

**Task 2.6 - Part B (computational).** Go back to the `res2` regression from Part 2 (`lsalary ~ lsales + lmktval + ceoten + college` on `ceosal2`). Read off `res2.rsquared` and `res2.rsquared_adj` - which is larger, and why must that always be the case (Section 3 of the CM)?


In [ ]:
print("R-squared:         ", res2.rsquared)
print("Adjusted R-squared:", res2.rsquared_adj)


## Part 3 — Your own analysis

You now repeat the same kind of analysis as Part 2, but on a dataset **you choose**, following the same steps. The point is not to write different code - it is to make your own decisions about which variables to use and why, and to interpret your own results.

### Pick a dataset

Choose **one** of the following (all from the `wooldridge` package, all suited to a *continuous* dependent variable - i.e., ordinary OLS, not yet the discrete-choice models of Session 4):

| Dataset | Topic | Size | A natural dependent variable |
|---|---|---|---|
| `hprice1` | House prices | 88 obs | `price` or `lprice` |
| `wage2` | Wages and worker characteristics | 935 obs | `wage` or `lwage` |
| `401ksubs` | Household wealth and retirement saving | 9,275 obs | `nettfa` (net financial assets) |
| `rdchem` | R&D spending in the chemical industry | 32 obs | `rd` or `lrd` |

Load the description of your chosen dataset with `load_data.data('name', description=True)` before deciding on your model - you need to know what the variables mean before choosing which ones to use.

### The canvas

Follow the same six steps as Part 2. Each step below has a markdown cell for your written answer and an empty code cell for you to fill in - reuse and adapt code from Part 2 as a starting point.

**Step 1 - Load and describe.** Load your dataset and look at summary statistics for your candidate variables.


**Step 2 - Choose your regressors.** Write your model down (dependent variable + 2 to 4 explanatory variables), and justify *economically* why each variable belongs in the model.

**Step 3 - Check your regressors before regressing.** Look at the distribution of your dependent variable (is a log transformation appropriate, as in Part 2 and TD 1?) and the correlation between your explanatory variables (any multicollinearity risk, as in 2.2?).

**Step 4 - Estimate.** Estimate your model with `statsmodels` and display the summary table.

**Step 5 - Test.** For each coefficient: is it statistically significant at the 5% level? State $H_0$ and $H_1$ for at least one of them explicitly.

**Step 6 - Interpret, and address one multivariate-specific question.** Give an economic interpretation of your main coefficient of interest (as in 2.4). Then pick **one** multivariate issue to investigate, as in 2.5:
- What happens to your main coefficient if you drop one of your other regressors? Is there evidence of omitted variable bias?
- *Or:* are any two of your regressors highly correlated? What does that imply for the precision of your estimates?
